In [2]:
import copy
import gc
import os
import sys
import time
from io import BytesIO
from pathlib import Path

import numpy as np
import open3d as o3d
import torch
import torch as th
import torch.nn as nn
import viser
import viser.transforms as tf
from PIL import Image
from plyfile import PlyData, PlyElement

from gui.utils import merge_meshes
from trellis.pipelines import TrellisTextTo3DPipeline
from trellis.utils import postprocessing_utils

# os.environ["SPCONV_ALGO"] = "native"  # Can be 'native' or 'auto', default is 'auto'.

In [3]:
class SpaceControl:
    def __init__(self):
        super().__init__()
        self.pipeline = TrellisTextTo3DPipeline.from_pretrained("gui")
        self.pipeline.cuda()

    def __call__(
        self,
        last_path: str = "gui/meshes/last.ply",
        output_path: str = "sample.glb",
        t0_idx=7,
        text_prompt: str = "shoe",
        image_prompt=None,
        steps=12,
        cfg_strength=7.5,
    ):
        # normalize last mesh
        last_path = Path(last_path)
        last_normalized_path = last_path.parent / f"{last_path.stem}_normalized{last_path.suffix}"

        mesh = o3d.io.read_triangle_mesh(str(last_path))
        aabb = mesh.get_axis_aligned_bounding_box()
        min_bound = aabb.get_min_bound()
        max_bound = aabb.get_max_bound()
        center = (min_bound + max_bound) / 2
        scale = 1.0 / (max_bound - min_bound).max()
        mesh.translate(-center)
        mesh.scale(scale, center=(0, 0, 0))
        o3d.io.write_triangle_mesh(str(last_normalized_path), mesh)

        # apply Trellis
        outputs = self.pipeline.run(
            text_prompt,
            image_prompt,
            seed=1,
            sparse_structure_sampler_params={
                "steps": steps,
                "cfg_strength": cfg_strength,
                "t0_idx_value": t0_idx,
                "spatial_control_mesh_path": str(last_normalized_path),
            },
        )

        # apply trimesh conversion
        glb = postprocessing_utils.to_glb(
            outputs["gaussian"][0],
            outputs["mesh"][0],
            # Optional parameters
            simplify=0.95,  # Ratio of triangles to remove in the simplification process
            texture_size=1024,  # Size of the texture used for the GLB
        )

        # rotate x-axis clock wise
        rot_matrix = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, -1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0]])

        # denormalize
        glb.apply_scale(1 / scale)
        glb.apply_translation(center)  # bring to original scale and position before saving
        glb.apply_transform(rot_matrix)

        # save as glb
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        glb.export(str(output_path))

        # clean
        gc.collect()
        th.cuda.empty_cache()

In [4]:
space_control = SpaceControl()

[SPARSE][CONV] spconv algo: auto
[ATTENTION] Using backend: flash_attn


Using cache found in /home/rvi/.cache/torch/hub/facebookresearch_dinov2_main
/home/rvi/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/rvi/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/rvi/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [5]:
space_control(output_path=f"results/sample_image1.glb", image_prompt=Image.open("gui/meshes/shoe1.jpg"))

Sampling: 100%|██████████| 12/12 [00:01<00:00,  9.05it/s]
100%|████████████████████████████████████████| 176M/176M [00:00<00:00, 546GB/s]
Sampling: 100%|██████████| 25/25 [00:02<00:00, 11.20it/s]


Before postprocess: 143208 vertices, 286428 faces


Decimating Mesh: 100%|██████████[00:00<00:00]


After decimate: 7154 vertices, 14320 faces


/home/rvi/conda/envs/torch/lib/python3.11/site-packages/torch/utils/cpp_extension.py:2356: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1889.05it/s]


Found 1637 invisible faces
Dual graph: 21478 edges
Mincut solved, start checking the cut
Removed 7049 faces by mincut
INFO- Loaded 3670 vertices and 7271 faces.

100% done 
After remove invisible faces: 3670 vertices, 7275 faces


Rendering: 100it [00:00, 235.83it/s]
Texture baking (opt): optimizing: 100%|██████████| 2500/2500 [00:06<00:00, 361.26it/s, loss=0.0248]


In [6]:
for i in range(1, 10):
    space_control(output_path=f"results/sample_{i:02d}.glb", t0_idx=i)

Sampling: 100%|██████████| 25/25 [00:08<00:00,  2.86it/s]


Before postprocess: 220452 vertices, 440932 faces


Decimating Mesh: 100%|██████████[00:01<00:00]


After decimate: 11007 vertices, 22046 faces


Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1701.33it/s]


...

Decimating Mesh: 100%|██████████[00:00<00:00]


After decimate: 7334 vertices, 14660 faces


Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1868.66it/s]


Found 10482 invisible faces
Dual graph: 21990 edges
Mincut solved, start checking the cut
Removed 10482 faces by mincut
INFO- Loaded 2091 vertices and 4178 faces.

0% done 
After remove invisible faces: 2091 vertices, 4178 faces


Rendering: 100it [00:00, 245.54it/s]
Texture baking (opt): optimizing: 100%|██████████| 2500/2500 [00:07<00:00, 350.51it/s, loss=0.0122] 
